# Running on Modal

## What you'll learn

- Configure `ModalComputeConfig` on an operation
- Route in-memory operations to Modal containers
- Route file-based operations with automatic sandbox transport
- Specify GPU type, memory, and timeout per step
- Debug Modal execution with validation and logging

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb),
[Compute Routing](13-compute-routing.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No (uses CPU Modal containers for demonstration).

:::{note}
This tutorial requires a Modal account and authentication token. Code cells
are shown for reference but are not executed in the docs build.
:::

In [ ]:
from __future__ import annotations

from artisan.execution.compute import validate_remote_execute
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.schemas.operation_config.compute import Compute, ModalComputeConfig
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("modal_execution")
DELTA_ROOT = env.delta_root

## Configuring Modal compute

`ModalComputeConfig` defines the container settings for routing execute()
to Modal:

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `image` | `str` | (required) | Container image for the Modal function |
| `gpu` | `str \| None` | `None` | GPU type (e.g. `"T4"`, `"A10G"`, `"A100"`, `"H100"`) |
| `memory_gb` | `int` | `8` | Container memory in GB |
| `timeout` | `int` | `3600` | Per-call timeout in seconds |
| `retries` | `int` | `3` | Retries on preemption |

The container image must have artisan installed — transport functions run
inside the container.

In [ ]:
config = Compute(
    active="modal",
    modal=ModalComputeConfig(
        image="python:3.12-slim",
        gpu="T4",
    ),
)

print(f"Active provider: {config.active}")
print(f"Available:       {config.available()}")
print(f"Modal config:    image={config.modal.image}, gpu={config.modal.gpu}")

## Building a mixed local/Modal pipeline

A common pattern is to run lightweight steps locally and route
compute-intensive steps to Modal:

| Step | Operation | Compute | Why |
|------|-----------|---------|-----|
| generate | `DataGenerator` | LOCAL | Fast — creates small test datasets |
| transform | `DataTransformer` | **MODAL** | Compute-intensive transformation |
| metrics | `MetricCalculator` | **MODAL** | Compute-intensive metric calculation |

The pipeline wires outputs to inputs the same way regardless of compute
target.

In [ ]:
pipeline = PipelineManager.create(
    name="modal_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute="local",
)
output = pipeline.output

### Generate data (local)

DataGenerator is fast — no reason to wait for Modal container startup.

In [ ]:
step0 = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    compute="local",
)
print(f"Generated {step0.succeeded_count} datasets locally")

### Transform data (Modal)

The only change: `compute="modal"`. The operation, inputs, and output
wiring stay identical. The framework serializes the operation via
cloudpickle, ships it to a Modal container, runs execute(), and returns
the result.

In [ ]:
step1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 2, "seed": 100},
    compute="modal",
)
print(f"Transformed {step1.succeeded_count} datasets on Modal")

In [ ]:
step2 = pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    compute="modal",
)
print(f"Computed metrics for {step2.succeeded_count} datasets on Modal")

In [ ]:
summary = pipeline.finalize()
print(
    f"Pipeline complete: {summary['total_steps']} steps, "
    f"success={summary['overall_success']}"
)
inspect_pipeline(DELTA_ROOT)

## GPU and resource configuration

Different steps may need different GPU types. Use the dict syntax to
override Modal config per step:

```python
# String shorthand — uses the operation's default ModalComputeConfig
pipeline.run(MyOp, inputs=..., compute="modal")

# Dict syntax — override specific fields for this step
pipeline.run(
    MyOp,
    inputs=...,
    compute={"active": "modal", "modal": {"gpu": "A100", "memory_gb": 32}},
)
```

The dict is applied via `model_copy(update=...)`, so unspecified fields
keep their defaults.

In [ ]:
# Inspect what the dict override produces
base = Compute(
    active="local",
    modal=ModalComputeConfig(image="python:3.12-slim", gpu="T4"),
)

overridden = base.model_copy(
    update={"active": "modal", "modal": ModalComputeConfig(
        image="python:3.12-slim", gpu="A100", memory_gb=32,
    )}
)

print(f"Before: active={base.active}, gpu={base.modal.gpu}")
print(f"After:  active={overridden.active}, gpu={overridden.modal.gpu}, "
      f"memory_gb={overridden.modal.memory_gb}")

## File-based vs in-memory operations

The router handles both operation patterns transparently:

| Pattern | `materialize` | What ships to Modal | Sandbox transport |
|---------|--------------|---------------------|-------------------|
| In-memory | `False` | Operation + ExecuteInput (cloudpickle) | Empty — no files |
| File-based | `True` | Operation + ExecuteInput + sandbox snapshot | Files round-trip as `dict[str, bytes]` |

You don't need to change anything — the router detects which mode applies
based on whether the sandbox contains files.

**Transport limit:** 50 MB per direction (input snapshot + tool files;
output snapshot). For larger data, put files on object storage (S3, GCS)
and pass URIs as operation parameters instead of materializing to the
sandbox.

## Tool shipping

Operations that wrap external programs via `ToolSpec` need their tools
available in the Modal container:

| Tool type | How it gets there |
|-----------|------------------|
| Python scripts (bundled with your package) | Shipped automatically by the router |
| External binaries (Rosetta, GROMACS, etc.) | Must be pre-installed in the container image |

The `ToolSpec.executable` path on the local machine must match the path
in the container for external binaries. For bundled scripts, the router
restores them at the original absolute path before execute() runs.

## Debugging remote execution

### Validate before deploying

Run `validate_remote_execute()` on your operations before switching to
`compute="modal"`. It catches the most common issues upfront.

In [ ]:
gen = DataGenerator()
trans = DataTransformer()
calc = MetricCalculator()

for op in [gen, trans, calc]:
    result = validate_remote_execute(op)
    print(f"{type(op).__name__}: validate_remote_execute = {result}")

### Common issues

| Problem | Cause | Fix |
|---------|-------|-----|
| `RuntimeError` from validation | Operation has unpicklable attributes (file handles, lambdas) | Remove or make attributes picklable |
| `ImportError` in container | Artisan not installed in container image | Add artisan to the image's requirements |
| Missing binary in container | External tool not in image | Install the binary in the Dockerfile |
| `ValueError: Snapshot is X MB` | Sandbox files exceed 50 MB limit | Use object storage URIs instead of materializing |
| Unexpected Docker/Apptainer wrapping | Environment not forced to local | The router handles this automatically — if you see it, file a bug |

### Develop locally first

The fastest debugging strategy: run with `compute="local"` until your
pipeline logic is correct, then switch to `"modal"` for production.
The same operation code runs in both modes — only the compute target
changes.

## Summary

| Concept | What it does |
|---------|-------------|
| `ModalComputeConfig` | Container image, GPU, memory, timeout, retries |
| `compute="modal"` | Route execute() to a Modal container |
| Dict override | Per-step GPU/memory tuning via `compute={"active": "modal", ...}` |
| Sandbox transport | Automatic file round-trip for file-based operations (50 MB limit) |
| Tool shipping | Python scripts ship automatically; external binaries need the image |
| `validate_remote_execute()` | Pre-flight check before deploying to Modal |

Operations, inputs, params, and output wiring are identical whether
execute() runs locally or on Modal. Results land in the same Delta Lake
tables regardless of compute target.

## Next steps

- [Compute Routing](13-compute-routing.ipynb) — Framework concepts, validation, and pipeline defaults
- [SLURM Execution](07-slurm-execution.ipynb) — Run operations on a SLURM cluster
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work